# 05.02 Logistic Regression Baseline (Patient-Grouped 10-Fold CV)

In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, brier_score_loss, log_loss, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay,
)
from sklearn.calibration import CalibrationDisplay
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_SEED = 42

def find_project_root() -> Path:
    cur = Path.cwd().resolve()
    for p in [cur] + list(cur.parents):
        if (p / "data").exists() and (p / "experiments").exists():
            return p
    return cur

PROJECT_ROOT = find_project_root()
MODELS_DIR = PROJECT_ROOT / "models" / "phase_05"
REPORTS_DIR = PROJECT_ROOT / "reports" / "phase_05"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

split = json.loads((MODELS_DIR / "phase05_split_and_schema.json").read_text(encoding="utf-8"))
df_auth = pd.read_csv(PROJECT_ROOT / split["dataset_authentic"])
df_syn = pd.read_csv(PROJECT_ROOT / split["dataset_synthetic"])

NUM_COLS = split["num_cols"]
CAT_COLS = split["cat_cols"]
FEATURE_COLS = split["feature_cols"]

df_tr_dev = df_auth.iloc[split["auth_train_indices"]].reset_index(drop=True)
X_dev = df_tr_dev[FEATURE_COLS].copy()
y_dev = df_tr_dev["target"].to_numpy(dtype=int)
folds = split["cv_folds"]

def compute_binary_metrics(y_true, y_prob, threshold=0.5):
    y_t = np.asarray(y_true, dtype=int)
    y_p = np.clip(np.asarray(y_prob, dtype=float), 1e-7, 1.0 - 1e-7)
    y_pred = (y_p >= threshold).astype(int)
    cm = confusion_matrix(y_t, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    bal_acc = 0.5 * (spec + rec)
    bins = np.linspace(0.0, 1.0, 11)
    ece = 0.0
    for i in range(10):
        m = (y_p >= bins[i]) & (y_p < bins[i + 1] if i < 9 else y_p <= bins[i + 1])
        if m.any():
            ece += (m.sum() / len(y_t)) * abs(y_t[m].mean() - y_p[m].mean())
    auc = float(roc_auc_score(y_t, y_p)) if len(np.unique(y_t)) == 2 else float("nan")
    return {
        "accuracy": float(accuracy_score(y_t, y_pred)),
        "precision": float(precision_score(y_t, y_pred, pos_label=1, zero_division=0)),
        "recall": float(rec),
        "specificity": float(spec),
        "balanced_accuracy": float(bal_acc),
        "f1": float(f1_score(y_t, y_pred, pos_label=1, zero_division=0)),
        "macro_f1": float(f1_score(y_t, y_pred, average="macro", zero_division=0)),
        "roc_auc": auc,
        "brier_score": float(brier_score_loss(y_t, y_p)),
        "log_loss": float(log_loss(y_t, y_p, labels=[0, 1])),
        "ece": float(ece),
        "confusion_matrix": cm.tolist(),
    }

def build_lr_pipe():
    prep = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM_COLS),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="UNRECORDED")), ("ohe", OneHotEncoder(handle_unknown="ignore", min_frequency=3))]), CAT_COLS),
    ])
    return Pipeline([("prep", prep), ("clf", LogisticRegression(C=0.5, max_iter=1000, class_weight="balanced", random_state=RANDOM_SEED))])

def run_cv(augment_synthetic=False):
    oof_probs = np.zeros(len(y_dev), dtype=float)
    fold_rows = []
    for f in folds:
        tr_i, va_i = f["train_indices"], f["val_indices"]
        X_tr, y_tr = X_dev.iloc[tr_i].copy(), y_dev[tr_i].copy()
        if augment_synthetic:
            tr_pts = set(df_tr_dev.iloc[tr_i]["patient_group_id"])
            va_pts = set(df_tr_dev.iloc[va_i]["patient_group_id"])
            syn_f = df_syn[df_syn["patient_group_id"].isin(tr_pts)]
            assert len(set(syn_f["patient_group_id"]) & va_pts) == 0
            if len(syn_f) > 0:
                X_tr = pd.concat([X_tr, syn_f[FEATURE_COLS]], ignore_index=True)
                y_tr = np.concatenate([y_tr, syn_f["target"].to_numpy(dtype=int)])
        pipe = build_lr_pipe()
        pipe.fit(X_tr, y_tr)
        probs = pipe.predict_proba(X_dev.iloc[va_i])[:, 1]
        oof_probs[va_i] = probs
        m = compute_binary_metrics(y_dev[va_i], probs)
        m["fold"] = f["fold"]
        fold_rows.append(m)
    overall = compute_binary_metrics(y_dev, oof_probs)
    return oof_probs, pd.DataFrame(fold_rows), overall

oof_a, folds_a, overall_a = run_cv(augment_synthetic=False)
oof_b, folds_b, overall_b = run_cv(augment_synthetic=True)

print("Track A (Authentic-Only) 10-Fold OOF:", json.dumps(overall_a, indent=2))
print("Track B (Synthetic-Augmented Train, Authentic Val) 10-Fold OOF:", json.dumps(overall_b, indent=2))

# Fit final Track A & Track B models on full train_dev
lr_final_a = build_lr_pipe().fit(X_dev, y_dev)
joblib.dump(lr_final_a, MODELS_DIR / "logistic_regression_baseline.joblib")

syn_tr = df_syn.iloc[split["syn_train_indices"]]
lr_final_b = build_lr_pipe().fit(
    pd.concat([X_dev, syn_tr[FEATURE_COLS]], ignore_index=True),
    np.concatenate([y_dev, syn_tr["target"].to_numpy(dtype=int)]),
)
joblib.dump(lr_final_b, MODELS_DIR / "logistic_regression_trackB.joblib")

np.save(MODELS_DIR / "lr_oof_probs.npy", oof_a)
np.save(MODELS_DIR / "lr_trackB_oof_probs.npy", oof_b)
folds_a.drop(columns=["confusion_matrix"]).to_csv(REPORTS_DIR / "02_logistic_regression_10fold_cv.csv", index=False)

# Feature coefficients
feat_names = list(lr_final_a.named_steps["prep"].get_feature_names_out())
coefs = lr_final_a.named_steps["clf"].coef_[0]
top_feats = sorted(
    [{"feature": f, "coef": float(c), "abs_coef": float(abs(c))} for f, c in zip(feat_names, coefs)],
    key=lambda x: x["abs_coef"], reverse=True,
)[:15]

payload = {
    "model_name": "Logistic Regression",
    "cv_protocol": "10-Fold StratifiedGroupKFold (Authentic Patients)",
    "n_train_dev_patients": int(len(y_dev)),
    "track_a_oof": overall_a,
    "track_a_fold_mean": folds_a.drop(columns=["fold", "confusion_matrix"]).mean().to_dict(),
    "track_a_fold_std": folds_a.drop(columns=["fold", "confusion_matrix"]).std().to_dict(),
    "track_b_oof": overall_b,
    "top_features": top_feats,
}
(MODELS_DIR / "lr_metrics.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")

# Plots (Track A OOF)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ConfusionMatrixDisplay.from_predictions(y_dev, (oof_a >= 0.5).astype(int), display_labels=["INELIGIBLE", "ELIGIBLE"], ax=ax, cmap="Greens")
ax.set_title("Logistic Regression — Track A 10-Fold OOF Confusion Matrix")
plt.tight_layout(); plt.savefig(REPORTS_DIR / "02_lr_confusion_matrix.png", dpi=200); plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
RocCurveDisplay.from_predictions(y_dev, oof_a, ax=axes[0]); axes[0].set_title("LR — Track A ROC Curve")
PrecisionRecallDisplay.from_predictions(y_dev, oof_a, ax=axes[1]); axes[1].set_title("LR — Track A PR Curve")
plt.tight_layout(); plt.savefig(REPORTS_DIR / "02_lr_roc_pr_curves.png", dpi=200); plt.close(fig)

fig, ax = plt.subplots(figsize=(5.5, 4.5))
CalibrationDisplay.from_predictions(y_dev, oof_a, n_bins=8, ax=ax)
ax.set_title("LR — Track A Calibration Curve")
plt.tight_layout(); plt.savefig(REPORTS_DIR / "02_lr_calibration_curve.png", dpi=200); plt.close(fig)
print("Saved LR artifacts and plots.")


Track A (Authentic-Only) 10-Fold OOF: {
  "accuracy": 0.9157894736842105,
  "precision": 0.9482758620689655,
  "recall": 0.9166666666666666,
  "specificity": 0.9142857142857143,
  "balanced_accuracy": 0.9154761904761904,
  "f1": 0.9322033898305084,
  "macro_f1": 0.9105461393596986,
  "roc_auc": 0.949047619047619,
  "brier_score": 0.07677292315531485,
  "log_loss": 0.25916540137753535,
  "ece": 0.063793255207894,
  "confusion_matrix": [
    [
      32,
      3
    ],
    [
      5,
      55
    ]
  ]
}
Track B (Synthetic-Augmented Train, Authentic Val) 10-Fold OOF: {
  "accuracy": 0.8842105263157894,
  "precision": 0.9152542372881356,
  "recall": 0.9,
  "specificity": 0.8571428571428571,
  "balanced_accuracy": 0.8785714285714286,
  "f1": 0.907563025210084,
  "macro_f1": 0.8763167238726477,
  "roc_auc": 0.9295238095238095,
  "brier_score": 0.09455918542219494,
  "log_loss": 0.3469687704053912,
  "ece": 0.07618783864062477,
  "confusion_matrix": [
    [
      30,
      5
    ],
    [
    

Saved LR artifacts and plots.
